# 10 Beta Diversity and Age

We test whether fungal community composition is associated with age in years (`age_years_sample`). We compare whole-ASV and 16-mer Bray–Curtis distances using the same samples for both representations and both models. The adjusted model tests age after accounting for settlement type and diet.

## 1. Setup

### 1.1 Imports

In [1]:
import pandas as pd
import qiime2 as q2
from skbio import DistanceMatrix
from qiime2 import Visualization

### 1.2 Notebook settings and paths

We define the input paths, the shared complete-case files, and the outputs for each feature representation and model.

In [23]:
# 1. Shared inputs and output directories
metadata_path = "../data/processed_ignored/06_diversity/Final_metadata_with_GHSL.tsv"
kmer16_distance_path = "../data/processed_ignored/07_beta_diversity/kmer16_bray_curtis_distance.qza"
asv_distance_path = "../data/processed_ignored/07_beta_diversity/asv_bray_curtis_distance.qza"

out_dir = "../data/processed_ignored/10_beta_diversity_age"
vis_dir = "../data/processed/10_beta_diversity_age"

columns = ["Settlement_Type", "age_years_sample", "diet_type_sample", "sex_sample"]


# 2. Complete-case inputs for both models
adjusted_age_out_dir = f"{out_dir}/adjusted_age"

adjusted_age_distance_path = f"{adjusted_age_out_dir}/asv_bray_curtis_complete.qza"
kmer16_complete_distance_path = f"{adjusted_age_out_dir}/kmer16_bray_curtis_complete.qza"
adjusted_age_metadata_path = f"{adjusted_age_out_dir}/metadata_complete.tsv"


# 3. Overall PERMANOVA outputs
unadjusted_permanova_dir = f"{vis_dir}/unadjusted_permanova"
adjusted_permanova_dir = f"{vis_dir}/adjusted_permanova"

asv_unadjusted_overall_qzv = f"{unadjusted_permanova_dir}/asv_overall.qzv"
kmer16_unadjusted_overall_qzv = f"{unadjusted_permanova_dir}/kmer16_overall.qzv"

asv_adjusted_overall_qzv = f"{adjusted_permanova_dir}/asv_overall.qzv"
kmer16_adjusted_overall_qzv = f"{adjusted_permanova_dir}/kmer16_overall.qzv"


# Create directories
!mkdir -p "$out_dir" "$vis_dir" "$adjusted_age_out_dir"
!mkdir -p "$unadjusted_permanova_dir" "$adjusted_permanova_dir"

### 1.3 Load inputs

We load the whole-ASV and 16-mer Bray–Curtis distance matrices from notebook 07 and the cleaned metadata from notebook 06.

In [4]:
metadata = q2.Metadata.load(metadata_path).to_dataframe()
distance_matrix = q2.Artifact.load(kmer16_distance_path).view(DistanceMatrix)
asv_matrix = q2.Artifact.load(asv_distance_path).view(DistanceMatrix)

print("Whole-ASV samples:", len(asv_matrix.ids))
print("16-mer samples:", len(distance_matrix.ids))
print("Metadata rows:", len(metadata))

Whole-ASV samples: 150
16-mer samples: 150
Metadata rows: 150


## 2. Select samples

### 2.1 Match sample IDs

We keep samples present in both Bray–Curtis matrices and in the metadata.

In [24]:
shared_ids = pd.Index(distance_matrix.ids).intersection(asv_matrix.ids)
shared_ids = shared_ids.intersection(metadata.index)

matched = metadata.loc[shared_ids, columns].copy()

print("Matched samples:", len(matched))

print(age_complete.columns.tolist())
print("Complete-case samples:", len(age_complete))

Matched samples: 150
['Settlement_Type', 'age_years_sample', 'diet_type_sample', 'sex_sample']
Complete-case samples: 139


### 2.2 Keep complete cases

We keep samples labeled Rural, Suburban, or Urban with a numeric age and recorded diet and sex. Both PERMANOVA models use this same subset.

In [25]:
matched["Settlement_Type"] = matched["Settlement_Type"].str.strip()
matched["diet_type_sample"] = matched["diet_type_sample"].str.strip().replace("", pd.NA)
matched["sex_sample"] = matched["sex_sample"].str.strip().replace("", pd.NA)
matched["age_years_sample"] = pd.to_numeric(matched["age_years_sample"], errors="coerce")

keep = matched["Settlement_Type"].isin(["Rural", "Suburban", "Urban"])
age_complete = matched.loc[keep].dropna(subset=columns)
complete_ids = age_complete.index.tolist()

print("Complete-case samples:", len(complete_ids))

Complete-case samples: 139


We subset both Bray–Curtis matrices to the complete-case sample IDs in the same order and save them with the matching metadata.

In [26]:
distance_complete = asv_matrix.filter(complete_ids)
kmer16_complete = distance_matrix.filter(complete_ids)

assert list(distance_complete.ids) == complete_ids
assert list(kmer16_complete.ids) == complete_ids

age_complete.index.name = "sample-id"
q2.Metadata(age_complete).save(adjusted_age_metadata_path)

q2.Artifact.import_data("DistanceMatrix", distance_complete).save(
    adjusted_age_distance_path
)
q2.Artifact.import_data("DistanceMatrix", kmer16_complete).save(
    kmer16_complete_distance_path
)

print("Saved both distance matrices and metadata for", len(complete_ids), "samples.")

Saved both distance matrices and metadata for 139 samples.


### 2.3 Summarize samples

We compare the number of samples with a numeric age before and after requiring complete metadata. We also summarize the age distribution among the complete cases.

In [19]:
age_samples = matched["age_years_sample"].dropna()

counts = pd.DataFrame({
    "With numeric age": [len(age_samples)],
    "With all four variables": [len(age_complete)],
}, index=["Samples"])

display(counts)
display(age_complete["age_years_sample"].describe())

,With numeric age,With all four variables
Samples,145,139


count    139.000000
mean      48.834532
std       18.668581
min        2.000000
25%       36.000000
50%       53.000000
75%       63.500000
max       82.000000
Name: age_years_sample, dtype: float64

## 3. Overall PERMANOVA

### 3.1 Unadjusted PERMANOVA

We test whether fungal community composition is associated with age in years. Both tests use the same 139 samples and include age as the only predictor.

#### 3.1.1 Whole ASVs

In [9]:
!qiime diversity adonis \
    --i-distance-matrix "$adjusted_age_distance_path" \
    --m-metadata-file "$adjusted_age_metadata_path" \
    --p-formula "age_years_sample" \
    --o-visualization "$asv_unadjusted_overall_qzv"

Saved Visualization to: ../data/processed/10_beta_diversity_age/unadjusted_permanova/asv_overall.qzv


In [ ]:
Visualization.load(asv_unadjusted_overall_qzv)

#### 3.1.2 16-mers

In [10]:
!qiime diversity adonis \
    --i-distance-matrix "$kmer16_complete_distance_path" \
    --m-metadata-file "$adjusted_age_metadata_path" \
    --p-formula "age_years_sample" \
    --o-visualization "$kmer16_unadjusted_overall_qzv"

Saved Visualization to: ../data/processed/10_beta_diversity_age/unadjusted_permanova/kmer16_overall.qzv


In [ ]:
Visualization.load(kmer16_unadjusted_overall_qzv)

#### Unadjusted results

| Age term | Whole ASVs | 16-mers |
| --- | ---: | ---: |
| Samples | 139 | 139 |
| R² | 0.0115 | 0.0166 |
| Pseudo-F | 1.589 | 2.311 |
| p-value | 0.082 | **0.026** |

Age explained a small proportion of variation in both tests (1.15% for whole ASVs and 1.66% for 16-mers). Only the 16-mer test was statistically significant at the 0.05 threshold.

### 3.2 Adjusted PERMANOVA

We put age last in the formula to test whether fungal community composition is associated with age in years after accounting for settlement type, diet and sex. Both tests use the same 139 samples as the unadjusted tests.

#### 3.2.1 Whole ASVs

In [27]:
!qiime diversity adonis \
    --i-distance-matrix "$adjusted_age_distance_path" \
    --m-metadata-file "$adjusted_age_metadata_path" \
    --p-formula "Settlement_Type + diet_type_sample + sex_sample + age_years_sample" \
    --o-visualization "$asv_adjusted_overall_qzv"

Saved Visualization to: ../data/processed/10_beta_diversity_age/adjusted_permanova/asv_overall.qzv


In [ ]:
Visualization.load(asv_adjusted_overall_qzv)

#### 3.2.2 16-mers

In [28]:
!qiime diversity adonis \
    --i-distance-matrix "$kmer16_complete_distance_path" \
    --m-metadata-file "$adjusted_age_metadata_path" \
    --p-formula "Settlement_Type + diet_type_sample + sex_sample + age_years_sample" \
    --o-visualization "$kmer16_adjusted_overall_qzv"

Saved Visualization to: ../data/processed/10_beta_diversity_age/adjusted_permanova/kmer16_overall.qzv


In [ ]:
Visualization.load(kmer16_adjusted_overall_qzv)

#### Adjusted results

| Age term | Whole ASVs | 16-mers |
| --- | ---: | ---: |
| Samples | 139 | 139 |
| R² | 0.0111 | 0.0149 |
| Pseudo-F | 1.546 | 2.079 |
| p-value | 0.084 | **0.030** |

After accounting for settlement type, diet, and sex, age explained 1.11% of the variation in fungal community composition for whole ASVs and 1.49% for 16-mers. Only the 16-mer test was statistically significant at the 0.05 threshold.

### 3.3 Compare unadjusted and adjusted PERMANOVA

All four tests used the same 139 samples. We compare the age term before and after accounting for settlement type and diet.

| Bray–Curtis representation | Model | Age R² | Pseudo-F | p-value |
| --- | --- | ---: | ---: | ---: |
| Whole ASVs | Unadjusted | 0.0115 | 1.589 | 0.082 |
| Whole ASVs | Adjusted | 0.0105 | 1.457 | 0.108 |
| 16-mers | Unadjusted | 0.0166 | 2.311 | **0.026** |
| 16-mers | Adjusted | 0.0148 | 2.051 | **0.046** |

The observed age R² decreased slightly after adjustment for both representations. The observed age R² was slightly higher for 16-mers than for whole ASVs in both models. At the 0.05 threshold, both 16-mer tests were statistically significant, and both whole-ASV tests had p-values above 0.05.

## 4. Discussion of overall PERMANOVA

The overall 16-mer tests provided evidence of an association between age and fungal community composition, with and without adjustment for settlement type and diet (p = 0.046 and 0.026, respectively). Both overall whole-ASV tests had p-values above 0.05. The evidence for an association therefore depended on the feature representation.

Age accounted for 1.05% to 1.66% of the variation in fungal community composition across the four tests. The observed age R² decreased slightly after adjustment for both representations.

These tests address whether fungal community composition varies with age. Whether Shannon diversity increases or decreases with age is a separate question about diversity within individual samples.